# Data loader demo (Project 2 – BRPMM)

How to use `src/data_loader.py` to load the process A / B batch time series for classification.

Each **batch** (`batch_id`) is one multivariate time series. Calibration batches are labelled `accepted` (0) or `rejected` (1); test batches are unlabelled.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))   # notebooks/ -> src/

import numpy as np
import matplotlib.pyplot as plt
from data_loader import load_variables, load_long, load_labels, load_dataset, Standardizer
from features import build_features, drop_constant

## 1. Variable names
Column names come from `team-workspace/variable_names/`.

In [ ]:
for p in ("A", "B"):
    print(p, len(load_variables(p)), load_variables(p))

## 2. Long format
`load_long` returns one row per time step: `batch_id`, `t` (step index), then the features.

In [ ]:
df = load_long("A", "calibration")
print(df.shape)
df.head()

In [ ]:
steps = df.groupby("batch_id").size()
steps.describe()

## 3. Labels

In [ ]:
labels = load_labels("A")
labels.value_counts().rename({0: "accepted", 1: "rejected"})

## 4. Model-ready arrays
`load_dataset` pads every batch to the longest series: `X` is *(batches, time steps, features)*. `lengths` and `mask` tell you which steps are real data.

In [ ]:
ds = load_dataset("A", "calibration")
print("X      ", ds.X.shape, ds.X.dtype)
print("y      ", ds.y.shape, "rejected =", ds.y.sum())
print("lengths", ds.lengths[:10])
print("mask   ", ds.mask.shape, "real steps =", ds.mask.sum())

### Accepted vs rejected batches
Plot a few batches per class for each feature (padding excluded).

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(ds.feature_names) // 3 + 1, 3, figsize=(15, 14))
for ax, (f, name) in zip(axes.flat, enumerate(ds.feature_names)):
    for cls, color in ((0, "tab:blue"), (1, "tab:red")):
        for i in rng.choice(np.flatnonzero(ds.y == cls), 5, replace=False):
            ax.plot(ds.X[i, : ds.lengths[i], f], color=color, alpha=0.4, lw=0.7)
    ax.set_title(name, fontsize=10)
for ax in axes.flat[len(ds.feature_names):]:
    ax.axis("off")
fig.legend(handles=[plt.Line2D([], [], color="tab:blue", label="accepted"),
                    plt.Line2D([], [], color="tab:red", label="rejected")], loc="lower right")
fig.tight_layout()

## 5. Standardisation
Fit on the training data (padding ignored), then apply the same scaling to the test data.

In [ ]:
test = load_dataset("A", "test")
sc = Standardizer().fit(ds)
ds_n, test_n = sc.transform(ds), sc.transform(test)

print("train mean/std (first 3 features):", ds_n.X[ds_n.mask].mean(0)[:3].round(3), ds_n.X[ds_n.mask].std(0)[:3].round(3))
print("test batches:", len(test_n), "labels:", test_n.y)

## 6. Early classification: keep only the first steps
`max_len` truncates each series (from the start by default; `align="end"` keeps the last steps instead).

In [ ]:
early = load_dataset("A", "calibration", max_len=500)
print(early.X.shape, "batches shorter than 500 steps:", (early.lengths < 500).sum())

## 7. Tabular features
`features.build_features` gives one row per batch: summary stats, trends, step-to-step dynamics and per-phase stats. `prefix_frac` uses only the start of each batch (early classification).

In [ ]:
feats = build_features(df)
feats_test = build_features(load_long("A", "test"))
feats, feats_test = drop_constant(feats, feats_test)
y = labels.loc[feats.index]
print(feats.shape, feats_test.shape, y.shape)
feats.head()

In [ ]:
early_feats = build_features(df, prefix_frac=0.3)   # first 30% of every batch
early_feats[["n_timesteps"]].head()

## 8. PyTorch
`to_torch()` returns a `TensorDataset` of `(X, lengths, y)` for a standard `DataLoader`. Use `subset()` to split by batch.

In [ ]:
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold

tr_idx, va_idx = next(StratifiedKFold(5, shuffle=True, random_state=0).split(ds.X[:, 0, 0], ds.y))
train, val = ds.subset(tr_idx), ds.subset(va_idx)
sc = Standardizer().fit(train)
train, val = sc.transform(train), sc.transform(val)

train_dl = DataLoader(train.to_torch(), batch_size=16, shuffle=True)
X, lengths, yb = next(iter(train_dl))
print(X.shape, lengths[:5], yb[:5])

# masking the padding, e.g. for a masked mean over time:
mask = torch.arange(X.shape[1])[None, :] < lengths[:, None]
pooled = (X * mask[..., None]).sum(1) / lengths[:, None]
print(pooled.shape)

## 9. Process B
Same API. Process B has 20 features, every batch is 5001 steps, and only 5 of 40 calibration batches are rejected.

In [ ]:
dsB = load_dataset("B", "calibration")
print(dsB.X.shape, "rejected =", dsB.y.sum(), "/", len(dsB))